In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
import matplotlib.pyplot as plt
from dustmaps.sfd import SFDQuery
from astropy.coordinates import SkyCoord
from sklearn.model_selection import train_test_split

In [2]:
def redshift_estimator(df):
    import joblib
    import pandas as pd
    def extract_features(df):
        """
        計算去紅化星等並提取 5 維度的特徵空間。
        """
        df = df.copy()
        bands = ['g', 'r', 'i', 'z', 'y']
        
        # 1. 去紅化計算
        sfd = SFDQuery()
        coords = SkyCoord(ra = df['raMean'], dec = df['decMean'], unit = (u.deg, u.deg), frame = 'icrs')
        ebv = sfd(coords)
        coefficient = [3.172, 2.271, 1.682, 1.322, 1.087]
        extinction = [i*ebv for i in coefficient]
        for b, extinc in zip(bands, extinction):
            if f'{b}Mean{MAG}Mag' in df.columns:
                df[f'dered_{b}'] = df[f'{b}Mean{MAG}Mag'] - extinc
            else:
                raise ValueError(f"缺少 {b} 波段的資料")
                
        # 2. 計算 5D 測光特徵空間
        features = pd.DataFrame()
        features['mag_r'] = df['dered_r']
        features['g-r'] = df['dered_g'] - df['dered_r']
        features['r-i'] = df['dered_r'] - df['dered_i']
        features['i-z'] = df['dered_i'] - df['dered_z']
        features['z-y'] = df['dered_z'] - df['dered_y']
        # for b in bands:
        #    features[f'{b}_err'] = df[f'{b}MeanApMagErr']

        return df, features
    def calculate_photometric_redshift_llr(train_features, train_z_spec, target_features, k=100):
        """
        實作包含 Outlier Rejection 的局部線性迴歸演算法 (LLR)。
        """
        # 1. 初始化 NearestNeighbors
        nn = NearestNeighbors(n_neighbors=k, algorithm='auto')
        nn.fit(train_features)
        
        z_phot_pred = np.zeros(len(target_features))
        z_phot_err = np.zeros(len(target_features))
        z_std = []
        # 2. 針對「每一個」目標星系進行預測
        for i, target_feat in enumerate(target_features):
            # 尋找 k=100 個最近鄰
            distances, indices = nn.kneighbors([target_feat])
            neighbor_idx = indices[0]
            neighbor_redshifts = train_z_spec.iloc[neighbor_idx].values
            z_std.append(np.std(neighbor_redshifts))
            # 3. 提取特徵與真實紅移
            X_neighbors = train_features[neighbor_idx]
            y_neighbors = train_z_spec.iloc[neighbor_idx].values
            
            # 4. 首次擬合局部超平面
            lr = LinearRegression()
            lr.fit(X_neighbors, y_neighbors)
            
            # 5. Outlier Rejection Loop (剔除 > 3 sigma 殘差的鄰居)
            predictions = lr.predict(X_neighbors)
            residuals = np.abs(y_neighbors - predictions)
            std_residual = np.std(residuals)
            valid_mask = residuals < (3 * std_residual)
            
            # 防呆：如果剔除後鄰居過少，則不進行剔除
            if np.sum(valid_mask) > 10:
                X_clean, y_clean = X_neighbors[valid_mask], y_neighbors[valid_mask]
            else:
                X_clean, y_clean = X_neighbors, y_neighbors
                
            # 6. 利用剩下的鄰居重新擬合 (Re-fit)
            lr.fit(X_clean, y_clean)
            
            # 7. 預測目標星系的 z_phot (確保紅移不會小於0)
            z_pred = max(lr.predict([target_feat])[0], 0.0)
            z_phot_pred[i] = z_pred
            
            # 8. 儲存局部標準差作為誤差估計
            final_residuals = y_clean - lr.predict(X_clean)
            z_phot_err[i] = np.std(final_residuals)
            
        z_std = np.array(z_std)
        return z_phot_pred, z_phot_err, z_std
    # 初始化 Flat Lambda-CDM 宇宙學模型
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

    def calculate_physical_parameters(df):
        # 避免紅移為 0 導致發光距離與對數運算錯誤，設定下限
        z_safe = np.clip(df['z_phot'].values, 1e-4, None)
        
        # 1. 計算發光距離 D_L (單位 Mpc) 並轉換為 pc
        df['D_L_Mpc'] = cosmo.comoving_distance(z_safe).value
        df['D_L_pc'] = df['D_L_Mpc'] * 1e6
        
        # 2. 計算絕對星等 (Absolute Magnitude)
        # 公式: M_r = m_r - 5*log10(D_L/10) - K_correction
        # 備註: 這裡暫時假設 K-correction = 0
        distance_modulus = 5 * np.log10(df['D_L_pc'] / 10)
        df['Absolute_Mag_r'] = df['dered_r'] - distance_modulus
        
        # 3. 計算相對通量 (Flux)
        # 基於標準 SDSS AB 零點系統轉換: Flux = 10 ** ((8.90 - m) / 2.5)
        df['Calculated_Flux'] = 10 ** ((8.90 - df['dered_r']) / 2.5)
        
        return df
    def plot_redshift_comparison(true_z, pred_z, pred_z_err=None):
        """
        繪製真實驗證紅移與推算測光紅移的比較圖，包含完整統計指標與殘差圖。
        """
        import seaborn as sns
        from scipy import stats
        import numpy as np
        import matplotlib.pyplot as plt
        from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
        
        # 過濾無效值 (NaN或無限大)
        valid_idx = np.isfinite(true_z) & np.isfinite(pred_z)
        x = true_z[valid_idx]
        y = pred_z[valid_idx]
        
        y_err = pred_z_err[valid_idx] if pred_z_err is not None else None
            
        # ================= 建立畫布與子圖 =================
        fig, (ax_main, ax_res) = plt.subplots(2, 1, figsize=(8, 10), 
                                            gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
        
        # ================= 1. 上半部主圖 (Main Plot) =================
        ax_main.errorbar(x, y, yerr=y_err, fmt='o', markersize=4, alpha=0.4, 
                        ecolor='lightblue', elinewidth=0.8, capsize=0, zorder=1, label='Galaxies')
        
        limits = [max(min(x), min(y)), min(max(x), max(y))]
        ax_main.plot(limits, limits, color='gray', linestyle='--', alpha=0.8, zorder=2, label='1:1 Line')
        
        sns.regplot(x=x, y=y, ax=ax_main, scatter=False, color='red', 
                    line_kws={'label': 'Weighted Fit', 'zorder': 3})
        
        # ================= 🌟 計算更多統計指標 🌟 =================
        # 天文定義的紅移偏差 (Normalized residual): dz = (z_phot - z_spec) / (1 + z_spec)
        dz_norm = (y - x) / (1 + x) 
        
        # 1. 天文常用指標
        sigma_nmad = 1.48 * np.median(np.abs(dz_norm - np.median(dz_norm)))  # 歸一化絕對中位差 (更能抵抗離群值的標準差)
        f_outlier = np.sum(np.abs(dz_norm) > 0.15) / len(x) * 100            # 離群值比例 (誤差大於 15% 的比例)
        bias = np.median(dz_norm)                                            # 系統性偏差 (Median Bias，看整體是高估還低估)
        
        # 2. 統計學/機器學習常用指標
        r_val, _ = stats.pearsonr(x, y)                        # Pearson 相關係數 (r)，越接近 1 代表越呈直線相關
        r2 = r2_score(x, y)                                    # 決定係數 (R^2)，完美預測為 1.0
        mse = (mean_squared_error(x, y))               # 均方根誤差 (MSE)，模型整體的絕對誤差大小
        mae = mean_absolute_error(x, y)                    # 平均絕對誤差 (MAE)，模型整體的絕對誤差大小
        
        stats_text = (f"Correlation ($r$) = {r_val:.3f}\n"
                    f"$R^2$ Score = {r2:.3f}\n"
                    f"MSE = {mse:.4f}\n"
                    f"Bias ($\\Delta z$) = {bias:.4f}\n"
                    f"MAE = {mae:.4f}\n"
                    f"$\\sigma_{{NMAD}}$ = {sigma_nmad:.4f}\n"
                    f"$f_{{outlier}} = {f_outlier:.1f}\\%$")
        
        # 標註統計資訊
        props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='lightgray')
        ax_main.text(0.95, 0.05, stats_text, transform=ax_main.transAxes, fontsize=11,
                    verticalalignment='bottom', horizontalalignment='right', bbox=props, zorder=4)
        
        ax_main.set_title('Photometric Redshift Performance', fontsize=14, fontweight='bold')
        ax_main.set_ylabel('$z_{phot}$ (Photometric Redshift)', fontsize=12)
        ax_main.legend(loc='upper left', fontsize=11)
        ax_main.grid(True, linestyle=':', alpha=0.6)
        
        # ================= 2. 下半部殘差圖 (Residual Plot) =================
        ax_res.errorbar(x, y - x, yerr=y_err, fmt='o', markersize=3, alpha=0.4, ecolor='lightblue', elinewidth=0.8)
        ax_res.axhline(0, color='gray', linestyle='--', linewidth=1.5)
        
        ax_res.axhline(0.15, color='gray', linestyle=':', alpha=0.7)
        ax_res.axhline(-0.15, color='gray', linestyle=':', alpha=0.7)
        
        ax_res.set_xlabel('$z_{spec}$ (Spectroscopic Redshift)', fontsize=12)
        ax_res.set_ylabel('$\\Delta z$ ($z_{phot} - z_{spec}$)', fontsize=12)
        ax_res.set_ylim(-0.4, 0.4)
        ax_res.grid(True, linestyle=':', alpha=0.6)
        
        plt.subplots_adjust(hspace=0.05)
        import glob
        plt.savefig(f'../fig/photo_z_ps1_{len(glob.glob('../fig/*.png'))}.png', dpi = 300)
        plt.show()
    
    df, df_feature = extract_features(df)


    llr_model_data = joblib.load('llr_model_package.pk1')
    scaler_loaded = llr_model_data['scaler']
    train_features_loaded = llr_model_data['train_features_scaled']
    train_z_loaded = llr_model_data['train_z_spec']
    k_loaded = llr_model_data['k']
    threshold_loaded = llr_model_data['threshold']

    df_feature_scaled = scaler_loaded.transform(df_feature)
    df['z_phot'], df['z_phot_err'], df['z_std']= calculate_photometric_redshift_llr(
        train_features_loaded,
        train_z_loaded,
        df_feature_scaled,
        k = k_loaded
    )
    df_filtered = df[df['z_std'] < threshold_loaded]
    print('finish prediction')

    df = calculate_physical_parameters(df)
    return df, df_filtered

In [4]:
MAG = 'Kron'
ps1filename = pd.read_csv('../data/merged_ps1.csv')
bands = ['g', 'r', 'i', 'z', 'y']
for b in bands:
    ps1filename = ps1filename[ps1filename[f'{b}Mean{MAG}Mag']!= -999.0]
    ps1filename = ps1filename[ps1filename[f'{b}Mean{MAG}MagErr']!= -999.0]
ps1filename = ps1filename[((ps1filename['rMeanApMag'] - ps1filename['rMeanPSFMag']) < 0)]

In [5]:
ps1filename, ps1file_filtered = redshift_estimator(ps1filename)

finish prediction


In [6]:
ps1filename.to_csv('ps1_estimate.csv')
ps1file_filtered.to_csv('ps1_estimate_filtered.csv')